# S14: Selección de características — NovaMarket

Este notebook compara tres enfoques sobre las 22 candidatas del catálogo de S13: filtro (información mutua y redundancia), wrapper (RFE con regresión logística) y embedded (importancia de Random Forest). La etiqueta `fuga_cliente` nunca forma parte de las predictoras; `id_cliente` se conserva solo para el archivo final.

La separación de entrenamiento/prueba ocurre antes de imputar, calcular correlaciones o ajustar selectores y agrupa por `id_cliente`, para que un cliente no aparezca en ambas particiones. Todo parámetro aprendido se ajusta con entrenamiento. La prueba se reserva para evaluación posterior y no interviene en la selección.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

WORKSPACE = Path.cwd().resolve()
if not (WORKSPACE / 'S13' / 'S13_dataset_con_features.csv').exists():
    WORKSPACE = WORKSPACE.parent
S14_DIR = WORKSPACE / 'S14'
S14_DIR.mkdir(exist_ok=True)
ruta_entrada = WORKSPACE / 'S13' / 'S13_dataset_con_features.csv'
df = pd.read_csv(ruta_entrada, encoding='utf-8-sig', dtype={'id_cliente': 'string'})

candidatas = [
    'antiguedad_cliente_dias', 'monto_por_unidad', 'pedidos_previos_cliente',
    'gasto_previo_cliente', 'ticket_promedio_previo_cliente', 'dias_desde_compra_previa',
    'rfm_score', 'precio_unitario', 'unidades_vendidas', 'monto_compra',
    'edad_cliente', 'peso_pedido_kg', 'es_venta_corporativa', 'mes_pedido',
    'es_fin_semana_pedido', 'longitud_comentario', 'num_palabras_comentario',
    'tiene_comentario', 'menciona_demora', 'puntos_fidelidad_actuales',
    'dias_reposicion', 'stock_minimo'
]
columnas_requeridas = set(candidatas) | {'fuga_cliente', 'id_cliente'}
faltantes = columnas_requeridas - set(df.columns)
assert not faltantes, f'Faltan columnas del catálogo S13: {sorted(faltantes)}'
X = df[candidatas].apply(pd.to_numeric, errors='coerce')
y = pd.to_numeric(df['fuga_cliente'], errors='raise').astype(int)
assert y.nunique() == 2, 'Se esperaban dos clases en fuga_cliente.'

particion = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
indices_train, indices_test = next(
    particion.split(X, y, groups=df['id_cliente'])
)
X_train_raw, X_test_raw = X.iloc[indices_train], X.iloc[indices_test]
y_train, y_test = y.iloc[indices_train], y.iloc[indices_test]
clientes_train = set(df['id_cliente'].iloc[indices_train].dropna())
clientes_test = set(df['id_cliente'].iloc[indices_test].dropna())
assert clientes_train.isdisjoint(clientes_test), 'Hay clientes compartidos entre entrenamiento y prueba.'
print(f'Filas: {len(df):,} | entrenamiento: {len(X_train_raw):,} | prueba: {len(X_test_raw):,}')
print('Distribución de fuga_cliente por partición:')
display(pd.DataFrame({
    'entrenamiento': y_train.value_counts(normalize=True),
    'prueba': y_test.value_counts(normalize=True),
}).rename_axis('fuga_cliente'))
print(f'Clientes únicos: entrenamiento={len(clientes_train):,}; prueba={len(clientes_test):,}; compartidos=0')
clientes_con_etiquetas_mixtas = df.groupby('id_cliente')['fuga_cliente'].nunique(dropna=False).gt(1).sum()
print(f'Control de etiqueta: {clientes_con_etiquetas_mixtas:,} clientes tienen ambas clases en sus pedidos; confirmar granularidad de fuga_cliente antes de modelar.')
print(f'Candidatas iniciales: {len(candidatas)}')

Filas: 551,790 | entrenamiento: 441,446 | prueba: 110,344
Distribución de fuga_cliente por partición:


,entrenamiento,prueba
fuga_cliente,,
0,0.797513,0.794524
1,0.202487,0.205476


Clientes únicos: entrenamiento=120,509; prueba=30,128; compartidos=0


Control de etiqueta: 74,938 clientes tienen ambas clases en sus pedidos; confirmar granularidad de fuga_cliente antes de modelar.
Candidatas iniciales: 22


## 1. Filtro: varianza, información mutua y redundancia

Se eliminan constantes y casi constantes (una sola observación distinta o una categoría que concentra al menos 99,5% del entrenamiento). La información mutua se estima en una muestra reproducible de hasta 100.000 filas de entrenamiento; luego se recorren las candidatas de mayor a menor puntuación y se descarta una si su correlación absoluta de Pearson con una ya aceptada supera 0,85. Ni los umbrales ni las puntuaciones usan la prueba.


In [2]:
imputador = SimpleImputer(strategy='median', keep_empty_features=True)
X_train = pd.DataFrame(
    imputador.fit_transform(X_train_raw), columns=candidatas, index=X_train_raw.index
)
X_test = pd.DataFrame(
    imputador.transform(X_test_raw), columns=candidatas, index=X_test_raw.index
)

def es_casi_constante(serie):
    frecuencias = serie.value_counts(normalize=True, dropna=False)
    return serie.nunique(dropna=False) <= 1 or (not frecuencias.empty and frecuencias.iloc[0] >= 0.995)

descartadas_varianza = [col for col in candidatas if es_casi_constante(X_train[col])]
disponibles = [col for col in candidatas if col not in descartadas_varianza]
assert disponibles, 'El filtro de varianza eliminó todas las candidatas.'

rng = np.random.default_rng(42)
indices_mi = X_train.index.to_numpy()
if len(indices_mi) > 100_000:
    indices_mi = rng.choice(indices_mi, size=100_000, replace=False)
X_mi = X_train.loc[indices_mi, disponibles]
y_mi = y_train.loc[indices_mi]
discretas = [i for i, col in enumerate(disponibles) if X_train[col].nunique() <= 10]
mi = mutual_info_classif(
    X_mi, y_mi, discrete_features=[i in discretas for i in range(len(disponibles))],
    random_state=42, n_neighbors=3
)
mi_score = pd.Series(mi, index=disponibles, name='informacion_mutua')
corr_train = X_train[disponibles].corr().abs()

seleccion_filtro = []
for feature in mi_score.sort_values(ascending=False, kind='stable').index:
    if all(corr_train.loc[feature, elegida] <= 0.85 for elegida in seleccion_filtro):
        seleccion_filtro.append(feature)
    if len(seleccion_filtro) == 10:
        break
print('Casi constantes descartadas:', descartadas_varianza or 'ninguna')
print('Features elegidas por filtro:', seleccion_filtro)
display(mi_score.sort_values(ascending=False).to_frame())

Casi constantes descartadas: ninguna
Features elegidas por filtro: ['antiguedad_cliente_dias', 'monto_compra', 'puntos_fidelidad_actuales', 'monto_por_unidad', 'dias_desde_compra_previa', 'longitud_comentario', 'mes_pedido', 'unidades_vendidas', 'ticket_promedio_previo_cliente', 'pedidos_previos_cliente']


,informacion_mutua
antiguedad_cliente_dias,0.006389
monto_compra,0.004773
puntos_fidelidad_actuales,0.004246
monto_por_unidad,0.002944
dias_desde_compra_previa,0.002595
precio_unitario,0.002592
longitud_comentario,0.002244
mes_pedido,0.001977
unidades_vendidas,0.001360
ticket_promedio_previo_cliente,0.001101


## 2. Wrapper: RFE

RFE elimina iterativamente características usando los coeficientes de una regresión logística balanceada. Se ajusta el escalador solo con entrenamiento (necesario por las distintas escalas) y se conservan las 8 características con mayor rango RFE. La muestra de ajuste, si se requiere, también se extrae únicamente del entrenamiento.


In [3]:
if len(disponibles) < 8:
    n_wrapper = len(disponibles)
else:
    n_wrapper = 8
X_fit = X_train[disponibles]
y_fit = y_train
if len(X_fit) > 150_000:
    indices_fit = X_fit.sample(n=150_000, random_state=42).index
    X_fit, y_fit = X_fit.loc[indices_fit], y_train.loc[indices_fit]
escalador_rfe = StandardScaler()
X_fit_escalado = escalador_rfe.fit_transform(X_fit)
modelo_rfe = LogisticRegression(
    solver='liblinear', class_weight='balanced', max_iter=300, random_state=42
)
rfe = RFE(estimator=modelo_rfe, n_features_to_select=n_wrapper, step=1)
rfe.fit(X_fit_escalado, y_fit)
seleccion_wrapper = [col for col, elegido in zip(disponibles, rfe.support_) if elegido]
rango_rfe = pd.Series(rfe.ranking_, index=disponibles, name='rango_rfe')
print('Features elegidas por RFE:', seleccion_wrapper)
display(rango_rfe.sort_values().to_frame())

Features elegidas por RFE: ['monto_por_unidad', 'precio_unitario', 'unidades_vendidas', 'monto_compra', 'edad_cliente', 'es_venta_corporativa', 'puntos_fidelidad_actuales', 'stock_minimo']


,rango_rfe
monto_por_unidad,1
precio_unitario,1
monto_compra,1
unidades_vendidas,1
es_venta_corporativa,1
edad_cliente,1
stock_minimo,1
puntos_fidelidad_actuales,1
num_palabras_comentario,2
menciona_demora,3


## 3. Embedded: importancia de Random Forest

El bosque aleatorio aprende las importancias durante el ajuste, con pesos balanceados. Se ordenan por importancia y se conservan las 8 primeras. El ajuste usa exclusivamente entrenamiento y una muestra máxima reproducible de 150.000 filas.


In [4]:
X_rf = X_train[disponibles]
y_rf = y_train
if len(X_rf) > 150_000:
    indices_rf = X_rf.sample(n=150_000, random_state=43).index
    X_rf, y_rf = X_rf.loc[indices_rf], y_train.loc[indices_rf]
modelo_rf = RandomForestClassifier(
    n_estimators=150, max_depth=14, min_samples_leaf=20,
    class_weight='balanced_subsample', n_jobs=-1, random_state=42
)
modelo_rf.fit(X_rf, y_rf)
importancia_rf = pd.Series(
    modelo_rf.feature_importances_, index=disponibles, name='importancia_random_forest'
).sort_values(ascending=False, kind='stable')
seleccion_embedded = list(importancia_rf.head(min(8, len(disponibles))).index)
print('Features elegidas por Random Forest:', seleccion_embedded)
display(importancia_rf.to_frame())

Features elegidas por Random Forest: ['monto_compra', 'edad_cliente', 'precio_unitario', 'monto_por_unidad', 'puntos_fidelidad_actuales', 'ticket_promedio_previo_cliente', 'peso_pedido_kg', 'gasto_previo_cliente']


,importancia_random_forest
monto_compra,0.148348
edad_cliente,0.102932
precio_unitario,0.100634
monto_por_unidad,0.099832
puntos_fidelidad_actuales,0.064081
ticket_promedio_previo_cliente,0.063533
peso_pedido_kg,0.063336
gasto_previo_cliente,0.061406
dias_desde_compra_previa,0.057799
antiguedad_cliente_dias,0.049366


## 4. Comparación y selección final

La tabla de coincidencias muestra la presencia de cada candidata en cada método. La selección final prioriza primero las que coinciden en más métodos y, en empate, las de mayor información mutua de entrenamiento. Se agregan candidatas hasta tener al menos 6 y se detiene en 8; en cada paso se exige que no exista correlación absoluta superior a 0,85 con las ya elegidas. Se informa explícitamente si una candidata quedó excluida por redundancia.


In [5]:
comparacion = pd.DataFrame(index=candidatas)
comparacion.index.name = 'feature'
comparacion['Filtro'] = comparacion.index.isin(seleccion_filtro)
comparacion['Wrapper_RFE'] = comparacion.index.isin(seleccion_wrapper)
comparacion['Embedded_RF'] = comparacion.index.isin(seleccion_embedded)
comparacion['Metodos'] = comparacion[['Filtro', 'Wrapper_RFE', 'Embedded_RF']].sum(axis=1)
comparacion['MI_entrenamiento'] = mi_score.reindex(comparacion.index).fillna(0)
display(comparacion.sort_values(['Metodos', 'MI_entrenamiento'], ascending=False))

orden_final = comparacion.sort_values(
    ['Metodos', 'MI_entrenamiento'], ascending=False, kind='stable'
).index.tolist()
seleccion_final = []
excluidas_redundancia = []
for feature in orden_final:
    if feature in descartadas_varianza:
        continue
    if all(corr_train.loc[feature, elegida] <= 0.85 for elegida in seleccion_final):
        seleccion_final.append(feature)
        if len(seleccion_final) == 8:
            break
    else:
        excluidas_redundancia.append(feature)
assert 6 <= len(seleccion_final) <= 10, f'Selección final fuera de rango: {len(seleccion_final)}'
pares_redundantes = [
    (seleccion_final[i], seleccion_final[j], corr_train.loc[seleccion_final[i], seleccion_final[j]])
    for i in range(len(seleccion_final)) for j in range(i + 1, len(seleccion_final))
    if corr_train.loc[seleccion_final[i], seleccion_final[j]] > 0.85
]
assert not pares_redundantes, f'Pares redundantes en selección final: {pares_redundantes}'

columnas_salida = seleccion_final + ['fuga_cliente', 'id_cliente']
ruta_salida = S14_DIR / 'novamarket_features_seleccionadas.csv'
df[columnas_salida].to_csv(ruta_salida, index=False, encoding='utf-8-sig')
print('Selección final:', seleccion_final)
print('Candidatas descartadas por redundancia:', excluidas_redundancia or 'ninguna')
for feature in excluidas_redundancia:
    correlaciones = corr_train.loc[feature, seleccion_final].sort_values(ascending=False)
    pareja = correlaciones.index[0]
    print(f'{feature} excluida por correlación con {pareja}: r={correlaciones.iloc[0]:.4f}')
print('Correlación absoluta máxima entre seleccionadas:', max(
    [corr_train.loc[seleccion_final[i], seleccion_final[j]]
     for i in range(len(seleccion_final)) for j in range(i + 1, len(seleccion_final))]
    or [0.0]
))
print(f'Dataset exportado: {ruta_salida} | forma: {df[columnas_salida].shape}')
print('Columnas exportadas:', columnas_salida)

,Filtro,Wrapper_RFE,Embedded_RF,Metodos,MI_entrenamiento
feature,,,,,
monto_compra,True,True,True,3,0.004773
puntos_fidelidad_actuales,True,True,True,3,0.004246
monto_por_unidad,True,True,True,3,0.002944
precio_unitario,False,True,True,2,0.002592
unidades_vendidas,True,True,False,2,0.001360
ticket_promedio_previo_cliente,True,False,True,2,0.001101
edad_cliente,False,True,True,2,0.000846
antiguedad_cliente_dias,True,False,False,1,0.006389
dias_desde_compra_previa,True,False,False,1,0.002595


Selección final: ['monto_compra', 'puntos_fidelidad_actuales', 'monto_por_unidad', 'unidades_vendidas', 'ticket_promedio_previo_cliente', 'edad_cliente', 'antiguedad_cliente_dias', 'dias_desde_compra_previa']
Candidatas descartadas por redundancia: ['precio_unitario']
precio_unitario excluida por correlación con monto_por_unidad: r=0.9944
Correlación absoluta máxima entre seleccionadas: 0.6832104568888346
Dataset exportado: C:\Users\L0s3r\OneDrive\Documentos\GitHub\pre_primera_clase\S14\novamarket_features_seleccionadas.csv | forma: (551790, 10)
Columnas exportadas: ['monto_compra', 'puntos_fidelidad_actuales', 'monto_por_unidad', 'unidades_vendidas', 'ticket_promedio_previo_cliente', 'edad_cliente', 'antiguedad_cliente_dias', 'dias_desde_compra_previa', 'fuga_cliente', 'id_cliente']
